# Langkah 11 — Pelatihan dan Perbandingan Model Machine Learning
**Klasifikasi air terbuka, eceng gondok, dan KJA di Waduk Saguling (Sentinel-2, fitur PCD)**

Notebook ini berjalan di **Kaggle** dan **Google Colab** tanpa perubahan kode.

| Lingkungan | Cara menyiapkan data |
| --- | --- |
| Kaggle | *Add Input* → *Upload* → unggah CSV `dataset_T1_v2.csv`, `dataset_T2_v2.csv`, `dataset_D1_v2.csv`, `dataset_D2_v2.csv`, `dataset_D3_v2.csv`, `dataset_T3_v2.csv` (dan opsional `waduk_T3_v2.csv`) sebagai satu Dataset. Notebook mencarinya otomatis di `/kaggle/input`. |
| Colab | CSV sudah ada di Google Drive folder `PCD_Saguling` (hasil Langkah 10). Notebook akan meminta izin *mount* Drive. |

Hasil (tabel CSV, gambar PNG, model) disimpan ke `hasil_ml/`. Di Kaggle unduh dari tab *Output*; di Colab tersimpan di Drive.

**Urutan eksperimen** (lihat `docs/metodologi-ml.md`):
**Data:** latih = petak A–C pada T1, T2 + petak D (D1), E (D2), F (D3); uji = petak A–C pada T3.

**Pemilihan terbaik memakai data latih saja**: model ML dari skor validasi silang, aturan PCD dari skor pada data latih. Skor T3 hanya dilaporkan, agar data uji tidak ikut menentukan keputusan apa pun.

0. **E0 Pembanding PCD tiga kelas**: lima aturan tanpa ML (R1–R5), ambang Otsu dari data latih.
1. **E1 Perbandingan model**: 5 model × 5 set fitur (FRGB, F0, F01, F012, F0123), dilatih pada semua data latih, diuji di T3.
2. **E2 Ablasi fitur PCD**: kelompok fitur bertambah (FRGB → F0 → F0123) untuk setiap model.
3. **E3 Uji statistik**: empat uji McNemar (koreksi Holm) dan interval kepercayaan bootstrap per blok.
4. **E4 Analisis**: matriks konfusi, kepentingan fitur, uji silang antartanggal, pembanding PCD.
5. **E5 Petak latih tambahan**: model terbaik dilatih dengan dan tanpa petak D–F, lalu dibandingkan di T3.
6. **(Opsional) Peta dan luas seluruh waduk.**

Jalankan **Runtime → Run all**. Waktu jalan biasanya beberapa menit; SVM adalah yang paling lama.

## 0. Parameter

In [ ]:
# ====== PARAMETER YANG BOLEH DIUBAH ======
VERSI_DATASET = 'v2'          # akhiran CSV dari Langkah 10 (v2 = label voting v2)
LATIH = ['T1', 'T2', 'D1', 'D2', 'D3']   # tanggal data latih (D1 = petak D, D2 = E, D3 = F)
PETAK_TAMBAHAN = ['D', 'E', 'F']    # petak latih tambahan (Langkah 05b)
UJI = 'T3'                    # tanggal data uji (tidak pernah dilihat saat pelatihan)
MIN_SEPAKAT_LATIH = 4         # data latih hanya piksel yang disepakati >= 4 dari 5 penilai
BUANG_EG_PETAK_C = True       # piksel eceng gondok petak C tidak dipakai melatih (ambigu di sekitar KJA)
UKURAN_BLOK_M = 200           # blok spasial minimum (10 x 10 piksel) untuk validasi silang dan bootstrap
BLOK_OTOMATIS = True          # True: blok diperbesar bila jangkauan autokorelasi label > UKURAN_BLOK_M
BATAS_KORELASI = 0.05         # jangkauan autokorelasi = jarak pertama saat korelasi < nilai ini
N_FOLD = 5                    # GroupKFold pada blok untuk penyetelan hiperparameter
SEEDS = [0, 1, 2, 3, 4]       # pengulangan untuk model acak (RF, HGB, MLP)
N_BOOT = 1000                 # ulangan bootstrap
N_PERMUTASI = 10              # ulangan permutation importance
MAKS_LATIH_PER_KELAS = None   # mis. 3000 untuk mempercepat; None = pakai semua
SEED_UTAMA = 0

## 1. Lingkungan dan lokasi data

In [ ]:
import os, sys, glob, json, time, warnings, platform
warnings.filterwarnings('ignore')

def deteksi_lingkungan():
    if os.environ.get('PCD_DATA_DIR'):                       # untuk uji lokal
        return 'lokal', os.environ['PCD_DATA_DIR'], os.environ.get('PCD_OUT_DIR', './hasil_ml')
    if os.path.exists('/kaggle/input'):
        return 'kaggle', '/kaggle/input', '/kaggle/working/hasil_ml'
    try:
        import google.colab  # noqa: F401
        from google.colab import drive
        drive.mount('/content/drive')
        d = '/content/drive/MyDrive/PCD_Saguling'
        return 'colab', d, d + '/hasil_ml'
    except ImportError:
        return 'lokal', './data', './hasil_ml'

LINGKUNGAN, DATA_DIR, OUT_DIR = deteksi_lingkungan()
os.makedirs(OUT_DIR, exist_ok=True)

def cari(nama):
    hasil = glob.glob(os.path.join(DATA_DIR, '**', nama), recursive=True)
    return sorted(hasil)[0] if hasil else None

print('Lingkungan :', LINGKUNGAN)
print('Folder data:', DATA_DIR)
print('Folder hasil:', OUT_DIR)

In [ ]:
import numpy as np, pandas as pd, sklearn, scipy
import matplotlib.pyplot as plt
from sklearn.base import BaseEstimator, ClassifierMixin, clone
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import GridSearchCV, GroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.ensemble import RandomForestClassifier, HistGradientBoostingClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import (accuracy_score, f1_score, cohen_kappa_score,
                             confusion_matrix, precision_recall_fscore_support)
from scipy.stats import chi2, binomtest

VERSI = {'python': platform.python_version(), 'numpy': np.__version__, 'pandas': pd.__version__,
         'scikit-learn': sklearn.__version__, 'scipy': scipy.__version__}
print(VERSI)

## 2. Memuat dan memeriksa dataset

In [ ]:
KELAS = {0: 'Air', 1: 'Eceng gondok', 2: 'KJA'}
F0 = ['B2', 'B3', 'B4', 'B8', 'B5', 'B6', 'B7', 'B8A', 'B11', 'B12']
F1 = ['ndwi', 'mndwi', 'ndvi']
F2 = ['sd_nir', 'glcm_kontras', 'glcm_homogen', 'glcm_entropi', 'kepadatan_tepi']
F3 = ['tophat_putih', 'tophat_hitam', 'luas_komponen', 'jarak_air']
KELOMPOK = {'F0': F0, 'F1': F1, 'F2': F2, 'F3': F3}
FRGB = ['B2', 'B3', 'B4']             # pembanding: hanya pita tampak (RGB)
SET_FITUR = {                         # E2: kelompok fitur bertambah
    'FRGB': FRGB,
    'F0': F0,
    'F01': F0 + F1,
    'F012': F0 + F1 + F2,
    'F0123': F0 + F1 + F2 + F3,
}
LENGKAP = 'F0123'
MODEL_URUT = ['LR', 'SVM', 'RF', 'HGB', 'MLP']
# ID baris di spreadsheet (tab 4_ML_CV): M01 = LR FRGB ... M25 = MLP F0123.
ID_ML = {(m, fs): f'M{i * len(MODEL_URUT) + j + 1:02d}'
         for i, fs in enumerate(SET_FITUR) for j, m in enumerate(MODEL_URUT)}

def muat(kode):
    p = cari(f'dataset_{kode}_{VERSI_DATASET}.csv')
    if p is None:
        return None
    d = pd.read_csv(p)
    print(f'{kode}: {len(d):,} baris dari {p}')
    return d

data = {k: muat(k) for k in sorted(set(LATIH + [UJI]))}
UJI_COBA = data.get(UJI) is None
if UJI_COBA:
    # Uji coba sebelum T3 dilabeli: T2 menjadi data uji. JANGAN dipakai untuk naskah.
    print(f'PERINGATAN: dataset {UJI} tidak ditemukan. Mode uji coba: T2 menjadi data uji.')
    LATIH, UJI = [k for k in LATIH if k != 'T2'], 'T2'
tanpa = [k for k in LATIH if k in ('D1', 'D2', 'D3') and data.get(k) is None]
if tanpa:
    print(f'PERINGATAN: {tanpa} belum ada; dilatih tanpa petak tambahan pada tanggal itu.')
    LATIH = [k for k in LATIH if k not in tanpa]
hilang = [k for k in LATIH + [UJI] if data.get(k) is None]
assert not hilang, f'CSV tidak ditemukan untuk {hilang} di {DATA_DIR}'
df = pd.concat([data[k] for k in LATIH + [UJI]], ignore_index=True)

wajib = ['kode', 'petak', 'x', 'y', 'label', 'sepakat', 'air_pcd', 'waduk_pcd'] + sum(KELOMPOK.values(), [])
kurang = [c for c in wajib if c not in df.columns]
assert not kurang, f'Kolom hilang: {kurang}'

# Transformasi log untuk dua fitur yang sangat menceng (tidak mengubah urutan nilai).
for c in ['luas_komponen', 'jarak_air']:
    df[c] = np.log1p(df[c].clip(lower=0))

# Blok spasial 200 m per petak per tanggal (dipakai GroupKFold dan bootstrap).
x0 = df.groupby('petak')['x'].transform('min'); y0 = df.groupby('petak')['y'].transform('min')
df['blok'] = (df['kode'] + '_' + df['petak'] + '_' +
              ((df['x'] - x0) // UKURAN_BLOK_M).astype(int).astype(str) + '_' +
              ((df['y'] - y0) // UKURAN_BLOK_M).astype(int).astype(str))

nama_label = {-1: 'tanpa kelas', 0: 'air', 1: 'eceng gondok', 2: 'KJA', 3: 'luar waduk'}
ringkas = (df.assign(kelas=df['label'].map(nama_label))
             .pivot_table(index=['kode', 'petak'], columns='kelas', values='x', aggfunc='count', fill_value=0))
display(ringkas) if 'display' in globals() else print(ringkas)
ringkas.to_csv(f'{OUT_DIR}/jumlah_piksel_per_kelas.csv')

## 2b. Jangkauan autokorelasi spasial (dasar ukuran blok)
Roberts dkk. (2017) menyatakan jarak pemisah minimum dalam validasi silang adalah jangkauan autokorelasi, dan blok sebaiknya lebih besar daripadanya. Sel ini mengukur korelogram label (indikator tiap kelas) dan dua fitur di dalam tiap petak-tanggal. Jangkauan = jarak pertama saat korelasi turun di bawah `BATAS_KORELASI`. Bila jangkauan label melebihi `UKURAN_BLOK_M`, blok diperbesar otomatis. Korelogram residu model diperiksa lagi setelah E1.

In [ ]:
def korelogram(d, kolom, maks=1000, langkah=40, n_pasang=40000, seed=0):
    # Korelasi rata-rata pasangan piksel per kelas jarak, dihitung di dalam tiap petak-tanggal.
    rng = np.random.RandomState(seed)
    tepi = np.arange(0, maks + langkah, langkah)
    jum = {c: np.zeros(len(tepi) - 1) for c in kolom}; n = np.zeros(len(tepi) - 1)
    for _, g in d.groupby(['kode', 'petak']):
        if len(g) < 50:
            continue
        xy = g[['x', 'y']].values.astype(float)
        i = rng.randint(0, len(g), n_pasang); j = rng.randint(0, len(g), n_pasang)
        jarak = np.hypot(*(xy[i] - xy[j]).T)
        b = np.digitize(jarak, tepi) - 1
        ok = (b >= 0) & (b < len(n)) & (jarak > 0)
        np.add.at(n, b[ok], 1)
        for c in kolom:
            v = g[c].values.astype(float)
            z = (v - v.mean()) / (v.std() + 1e-12)
            np.add.at(jum[c], b[ok], (z[i] * z[j])[ok])
    tengah = (tepi[:-1] + tepi[1:]) / 2
    return pd.DataFrame({c: jum[c] / np.maximum(n, 1) for c in kolom}, index=tengah)

def jangkauan(seri, batas=BATAS_KORELASI):
    di_bawah = seri[seri < batas]
    return float(di_bawah.index[0]) if len(di_bawah) else float('inf')

dlab = df[df['label'].isin([0, 1, 2])].copy()
for k, nama in KELAS.items():
    dlab[f'kelas_{k}'] = (dlab['label'] == k).astype(float)
kor = korelogram(dlab, [f'kelas_{k}' for k in KELAS] + ['ndvi', 'mndwi'])
rentang = {c: jangkauan(kor[c]) for c in kor.columns}
tabel_rentang = pd.Series(rentang, name='jangkauan_m')
display(tabel_rentang.to_frame()) if 'display' in globals() else print(tabel_rentang)
tabel_rentang.to_csv(f'{OUT_DIR}/autokorelasi_jangkauan.csv'); kor.round(4).to_csv(f'{OUT_DIR}/autokorelasi_korelogram.csv')

r_label = max(v for c, v in rentang.items() if c.startswith('kelas_') and np.isfinite(v)) if any(
    np.isfinite(v) for c, v in rentang.items() if c.startswith('kelas_')) else float('inf')
if BLOK_OTOMATIS and r_label > UKURAN_BLOK_M:
    baru = int(min(500, np.ceil(r_label / 100) * 100))
    print(f'Jangkauan autokorelasi label {r_label:.0f} m > blok {UKURAN_BLOK_M} m: blok diperbesar menjadi {baru} m.')
    if r_label > 500:
        print('PERINGATAN: jangkauan > 500 m; blok dibatasi 500 m agar tiap petak masih punya >= 4 blok. Laporkan.')
    UKURAN_BLOK_M = baru
    df['blok'] = (df['kode'] + '_' + df['petak'] + '_' +
                  ((df['x'] - x0) // UKURAN_BLOK_M).astype(int).astype(str) + '_' +
                  ((df['y'] - y0) // UKURAN_BLOK_M).astype(int).astype(str))
print(f'Ukuran blok dipakai: {UKURAN_BLOK_M} m; jarak petak latih tambahan ke petak uji >= 1000 m.')

fig, ax = plt.subplots(figsize=(6.5, 3.8))
for c in kor.columns:
    ax.plot(kor.index, kor[c], label=c.replace('kelas_', 'kelas ').replace('0', KELAS[0]).replace('1', KELAS[1]).replace('2', KELAS[2]))
ax.axhline(BATAS_KORELASI, color='grey', lw=0.8, ls=':')
ax.axvline(UKURAN_BLOK_M, color='black', lw=1, ls='--'); ax.text(UKURAN_BLOK_M + 10, 0.9, 'ukuran blok', fontsize=8)
ax.set_xlabel('Jarak (m)'); ax.set_ylabel('Korelasi'); ax.set_ylim(-0.2, 1); ax.grid(alpha=0.3)
ax.set_title('Korelogram label dan fitur di dalam petak'); ax.legend(fontsize=8, frameon=False)
fig.tight_layout(); fig.savefig(f'{OUT_DIR}/gbr_korelogram.png', dpi=300); plt.show()

## 3. Data latih dan data uji

In [ ]:
kelas_ok = df['label'].isin([0, 1, 2])
latih = df[df['kode'].isin(LATIH) & kelas_ok & (df['sepakat'] >= MIN_SEPAKAT_LATIH)].copy()
if BUANG_EG_PETAK_C:
    latih = latih[~((latih['petak'] == 'C') & (latih['label'] == 1))]
uji = df[(df['kode'] == UJI) & kelas_ok].copy()        # semua tingkat kesepakatan
assert latih['label'].nunique() == 3, 'Data latih tidak memuat ketiga kelas.'

if MAKS_LATIH_PER_KELAS:
    latih = (latih.groupby('label', group_keys=False)
                  .apply(lambda g: g.sample(min(len(g), MAKS_LATIH_PER_KELAS), random_state=SEED_UTAMA)))

TINGKAT = {'semua': uji['sepakat'] >= 3, 'min 4 dari 5': uji['sepakat'] >= 4, '5 dari 5': uji['sepakat'] == 5}
tabel_n = pd.DataFrame({
    'latih': latih['label'].map(KELAS).value_counts(),
    **{f'uji ({t})': uji.loc[m, 'label'].map(KELAS).value_counts() for t, m in TINGKAT.items()}
}).fillna(0).astype(int)
tabel_n.loc['Jumlah'] = tabel_n.sum()
print(f'Latih: {LATIH} (sepakat >= {MIN_SEPAKAT_LATIH}), uji: {UJI}')
print('Petak di data latih:', sorted(latih['petak'].unique()), '| petak di data uji:', sorted(uji['petak'].unique()))
display(tabel_n) if 'display' in globals() else print(tabel_n)
tabel_n.to_csv(f'{OUT_DIR}/jumlah_data_latih_uji.csv')

## 3b. E0: Pembanding PCD tiga kelas (aturan tanpa ML)
Aturan PCD memakai peta air terbaik Langkah 07 (`air_pcd`) lalu memisahkan eceng gondok dan KJA pada piksel bukan air. Semua ambang dihitung dengan **Otsu pada data latih saja**, tanpa melihat label, sehingga aturan ini setara dengan "PCD tanpa pembelajaran". Aturan terbaik dipilih dari F1 makro pada data latih, lalu dibandingkan dengan ML di E3.

| Aturan | Air | EG vs KJA (piksel bukan air) |
| --- | --- | --- |
| R1 | `air_pcd` | NDVI ≥ t_ndvi → EG, selain itu KJA |
| R2 | `air_pcd` | R1, lalu komponen kecil (0 < luas < t_luas) → KJA |
| R3 | `air_pcd` | R1, lalu tekstur tinggi (sd_nir ≥ t_sd) → KJA |
| R4 | `air_pcd` | R1, lalu tekstur tinggi DAN komponen kecil → KJA |
| R5 | gray < t_g1 (grayscale RGB) | gray ≥ t_g2 → KJA, selain itu EG (pembanding RGB) |

In [ ]:
def metrik(y, p):
    pr, rc, f1, _ = precision_recall_fscore_support(y, p, labels=[0, 1, 2], zero_division=0)
    return {'OA': accuracy_score(y, p), 'kappa': cohen_kappa_score(y, p),
            'F1_makro': f1_score(y, p, labels=[0, 1, 2], average='macro', zero_division=0),
            'F1_air': f1[0], 'F1_eg': f1[1], 'F1_kja': f1[2],
            'recall_air': rc[0], 'recall_eg': rc[1], 'recall_kja': rc[2]}

def ambang_otsu(v, bins=256):
    # Otsu pada histogram 256 kelas: ambang dengan varians antarkelas terbesar.
    v = np.asarray(v, float); v = v[np.isfinite(v)]
    if len(v) < 2:
        return np.nan
    v = np.clip(v, *np.percentile(v, [0.5, 99.5]))       # pencilan tidak melebarkan histogram
    if v.min() == v.max():
        return np.nan
    h, tepi = np.histogram(v, bins=bins)
    c = (tepi[:-1] + tepi[1:]) / 2
    w0 = np.cumsum(h); w1 = w0[-1] - w0
    m0 = np.cumsum(h * c) / np.maximum(w0, 1)
    m1 = ((h * c).sum() - np.cumsum(h * c)) / np.maximum(w1, 1)
    return float(c[np.argmax(w0 * w1 * (m0 - m1) ** 2)])

def gray(d):
    return 0.299 * d['B4'] + 0.587 * d['B3'] + 0.114 * d['B2']   # ITU-R BT.601

# Ambang dari data latih (label TIDAK dipakai).
bukan_air = latih[latih['air_pcd'] == 0]
AMBANG_PCD = {
    't_ndvi': ambang_otsu(bukan_air['ndvi']),
    't_luas': ambang_otsu(bukan_air.loc[bukan_air['luas_komponen'] > 0, 'luas_komponen']),   # skala log1p
    't_sd': ambang_otsu(bukan_air['sd_nir']),
    't_g1': ambang_otsu(gray(latih)),
}
AMBANG_PCD['t_g2'] = ambang_otsu(gray(latih)[gray(latih) >= AMBANG_PCD['t_g1']])

def aturan(d, r):
    t = AMBANG_PCD
    if r == 'R5':
        g = gray(d).values
        return np.where(g < t['t_g1'], 0, np.where(g >= t['t_g2'], 2, 1))
    air = d['air_pcd'].values == 1
    kja = d['ndvi'].values < t['t_ndvi']
    kecil = (d['luas_komponen'].values > 0) & (d['luas_komponen'].values < t['t_luas'])
    kasar = d['sd_nir'].values >= t['t_sd']
    if r == 'R2': kja = kja | kecil
    if r == 'R3': kja = kja | kasar
    if r == 'R4': kja = kja | (kasar & kecil)
    return np.where(air, 0, np.where(kja, 2, 1))

baris_pcd = []
for r in ['R1', 'R2', 'R3', 'R4', 'R5']:
    ml, mu = metrik(latih['label'].values, aturan(latih, r)), metrik(uji['label'].values, aturan(uji, r))
    baris_pcd.append({'aturan': r, **{f'latih_{k}': v for k, v in ml.items()},
                      **{f'uji_{k}': v for k, v in mu.items()}})
e0 = pd.DataFrame(baris_pcd).set_index('aturan')
e0['peringkat_latih'] = e0['latih_F1_makro'].rank(ascending=False, method='min').astype(int)
PCD_TERBAIK = e0['latih_F1_makro'].idxmax()            # dipilih dari data LATIH
pred_pcd = aturan(uji, PCD_TERBAIK)
print('Ambang (dari data latih):', {k: round(v, 4) for k, v in AMBANG_PCD.items()})
print('Aturan PCD terbaik menurut data latih:', PCD_TERBAIK)
kol_e0 = ['latih_F1_air', 'latih_F1_eg', 'latih_F1_kja', 'latih_F1_makro', 'latih_OA', 'latih_kappa',
          'peringkat_latih', 'uji_F1_air', 'uji_F1_eg', 'uji_F1_kja', 'uji_F1_makro', 'uji_OA', 'uji_kappa']
display(e0[kol_e0].round(4)) if 'display' in globals() else print(e0[kol_e0].round(4))
e0[kol_e0].round(4).to_csv(f'{OUT_DIR}/e0_pcd_tiga_kelas.csv')
pd.Series(AMBANG_PCD).round(6).to_csv(f'{OUT_DIR}/e0_ambang_pcd.csv')

## 4. Model yang dibandingkan
Lima model mewakili lima keluarga pengklasifikasi yang umum di penginderaan jauh. Setiap model diberi penanganan ketidakseimbangan kelas yang setara: bobot kelas seimbang, atau *oversampling* di dalam fold untuk MLP. Hiperparameter disetel dengan **GroupKFold per blok 200 m pada data latih saja**; data uji tidak pernah dipakai untuk penyetelan.

In [ ]:
class Oversample(BaseEstimator, ClassifierMixin):
    # Menyamakan jumlah sampel tiap kelas dengan oversampling acak, hanya saat fit.
    def __init__(self, estimator=None, random_state=0):
        self.estimator = estimator
        self.random_state = random_state
    def fit(self, X, y):
        rng = np.random.RandomState(self.random_state)
        y = np.asarray(y); n = np.bincount(y).max()
        idx = np.concatenate([rng.choice(np.where(y == k)[0], n, replace=True) for k in np.unique(y)])
        self.estimator_ = clone(self.estimator).fit(np.asarray(X)[idx], y[idx])
        self.classes_ = self.estimator_.classes_
        return self
    def predict(self, X):
        return self.estimator_.predict(X)
    def predict_proba(self, X):
        return self.estimator_.predict_proba(X)

def hgb(seed):
    try:
        return HistGradientBoostingClassifier(class_weight='balanced', random_state=seed)
    except TypeError:     # scikit-learn < 1.2
        return Oversample(HistGradientBoostingClassifier(random_state=seed), random_state=seed)

def bangun(nama, seed=0):
    if nama == 'LR':
        return Pipeline([('skala', StandardScaler()),
                         ('model', LogisticRegression(class_weight='balanced', max_iter=3000))])
    if nama == 'SVM':
        return Pipeline([('skala', StandardScaler()),
                         ('model', SVC(kernel='rbf', class_weight='balanced'))])
    if nama == 'RF':
        return RandomForestClassifier(n_estimators=300, class_weight='balanced', n_jobs=-1, random_state=seed)
    if nama == 'HGB':
        return hgb(seed)
    if nama == 'MLP':
        return Pipeline([('skala', StandardScaler()),
                         ('model', Oversample(MLPClassifier(max_iter=600, early_stopping=True,
                                                            random_state=seed), random_state=seed))])
    raise ValueError(nama)

def awalan_hgb():
    return 'estimator__' if isinstance(hgb(0), Oversample) else ''

GRID = {
    'LR':  {'model__C': [0.1, 1, 10]},
    'SVM': {'model__C': [1, 10, 100], 'model__gamma': ['scale', 0.05]},
    'RF':  {'max_features': ['sqrt', 0.33], 'min_samples_leaf': [1, 5]},
    'HGB': {awalan_hgb() + 'learning_rate': [0.05, 0.1], awalan_hgb() + 'max_leaf_nodes': [15, 31]},
    'MLP': {'model__estimator__hidden_layer_sizes': [(64,), (64, 32)],
            'model__estimator__alpha': [1e-4, 1e-3]},
}
ACAK = {'RF', 'HGB', 'MLP'}            # diulang dengan beberapa seed
NAMA_MODEL = {'LR': 'Regresi logistik', 'SVM': 'SVM (RBF)', 'RF': 'Random forest',
              'HGB': 'Gradient boosting (HGB)', 'MLP': 'MLP'}

## 5. E1 dan E2: Perbandingan model dan ablasi fitur

In [ ]:
from sklearn.metrics import make_scorer
# Skor validasi silang yang dicatat. Penyetelan memakai F1 makro (refit).
SKOR = {'F1_makro': 'f1_macro', 'OA': 'accuracy', 'kappa': make_scorer(cohen_kappa_score),
        **{f'F1_{n}': make_scorer(f1_score, labels=[k], average='macro', zero_division=0)
           for k, n in [(0, 'air'), (1, 'eg'), (2, 'kja')]}}

y_latih, g_latih, y_uji = latih['label'].values, latih['blok'].values, uji['label'].values
cv = GroupKFold(n_splits=min(N_FOLD, latih['blok'].nunique()))
baris, baris_tingkat, prediksi, param_terbaik, model_akhir, skor_cv = [], [], {}, {}, {}, {}
mulai = time.time()

for fs, kolom in SET_FITUR.items():
    X_latih, X_uji = latih[kolom].values, uji[kolom].values
    for nama in GRID:
        t0 = time.time()
        gs = GridSearchCV(bangun(nama, SEED_UTAMA), GRID[nama], scoring=SKOR, refit='F1_makro',
                          cv=cv, n_jobs=-1)
        gs.fit(X_latih, y_latih, groups=g_latih)
        param_terbaik[(nama, fs)] = gs.best_params_
        i = gs.best_index_
        skor_cv[(nama, fs)] = {**{f'cv_{k}': gs.cv_results_[f'mean_test_{k}'][i] for k in SKOR},
                               # SD antarlipatan GroupKFold (bukan antarseed)
                               'cv_F1_makro_sd': gs.cv_results_['std_test_F1_makro'][i],
                               'waktu_dtk': time.time() - t0}
        for seed in (SEEDS if nama in ACAK else [SEED_UTAMA]):
            m = bangun(nama, seed).set_params(**gs.best_params_).fit(X_latih, y_latih)
            p = m.predict(X_uji)
            baris.append({'model': nama, 'fitur': fs, 'seed': seed, 'cv_F1_makro': gs.best_score_, **metrik(y_uji, p)})
            if seed == SEED_UTAMA:
                prediksi[(nama, fs)] = p
                model_akhir[(nama, fs)] = m
                for t, msk in TINGKAT.items():
                    baris_tingkat.append({'model': nama, 'fitur': fs, 'tingkat': t, 'n': int(msk.sum()),
                                          **metrik(y_uji[msk.values], p[msk.values])})
        print(f'{fs:6s} {nama:4s} selesai  cv F1={gs.best_score_:.3f}  ({time.time() - mulai:.0f} dtk)')

# Tabel siap salin ke spreadsheet tab 4_ML_CV (urutan baris M01-M25).
tab_cv = pd.DataFrame([{'id': ID_ML[(m, fs)], 'model': m, 'set_fitur': fs, 'jumlah_fitur': len(SET_FITUR[fs]),
                        **skor_cv[(m, fs)], 'hiperparameter': str(param_terbaik[(m, fs)])}
                       for fs in SET_FITUR for m in MODEL_URUT]).set_index('id')
tab_cv['peringkat_cv'] = tab_cv['cv_F1_makro'].rank(ascending=False, method='min').astype(int)
tab_cv.round(4).to_csv(f'{OUT_DIR}/e1e2_cv_untuk_spreadsheet.csv')

hasil = pd.DataFrame(baris)
kolom_m = ['OA', 'kappa', 'F1_makro', 'F1_air', 'F1_eg', 'F1_kja']
ringkasan = (hasil.groupby(['fitur', 'model'])[kolom_m + ['cv_F1_makro']]
                  .agg(['mean', 'std']).round(4))
hasil.to_csv(f'{OUT_DIR}/e1e2_hasil_per_seed.csv', index=False)
ringkasan.to_csv(f'{OUT_DIR}/e1e2_ringkasan_mean_sd.csv')
pd.DataFrame(baris_tingkat).round(4).to_csv(f'{OUT_DIR}/e1e2_per_tingkat_kesepakatan.csv', index=False)
pd.Series({f'{k[0]}|{k[1]}': v for k, v in param_terbaik.items()}).to_csv(f'{OUT_DIR}/hiperparameter_terbaik.csv')
print('Selesai dalam', round(time.time() - mulai), 'detik')

In [ ]:
# Tabel utama (fitur lengkap F0123) — Tabel IV Naskah 2
utama = (hasil[hasil['fitur'] == LENGKAP].groupby('model')[kolom_m].mean()
         .sort_values('F1_makro', ascending=False).round(4))
utama.index = [NAMA_MODEL[m] for m in utama.index]
display(utama) if 'display' in globals() else print(utama)
# Kombinasi model x set fitur terbaik dipilih dari skor VALIDASI SILANG pada data latih
# (GroupKFold per blok) dari ke-25 kombinasi, BUKAN dari skor T3, agar data uji tidak ikut
# menentukan model yang dianalisis lanjut.
urut_cv = tab_cv.sort_values('cv_F1_makro', ascending=False)
TERBAIK, SET_TERBAIK = urut_cv.iloc[0][['model', 'set_fitur']]
KEDUA = tuple(urut_cv.iloc[1][['model', 'set_fitur']])           # peringkat 2 menurut CV
# Pembanding RGB: kombinasi FRGB dengan CV tertinggi.
RGB_TERBAIK = tuple(urut_cv[urut_cv['set_fitur'] == 'FRGB'].iloc[0][['model', 'set_fitur']])
KUNCI = (TERBAIK, SET_TERBAIK)
print(f'Terbaik menurut CV: {NAMA_MODEL[TERBAIK]} + {SET_TERBAIK} ({ID_ML[KUNCI]}); '
      f'kedua: {KEDUA[0]} + {KEDUA[1]} ({ID_ML[KEDUA]}); RGB terbaik: {RGB_TERBAIK[0]} ({ID_ML[RGB_TERBAIK]})')

# Deteksi overfitting: selisih F1 makro validasi silang (data latih) dan uji (T3).
# Selisih besar = model terlalu menyesuaikan diri dengan data latih, atau T3 jauh berbeda.
celah = (hasil.groupby(['fitur', 'model'])[['cv_F1_makro', 'F1_makro']].mean()
              .rename(columns={'cv_F1_makro': 'F1_validasi', 'F1_makro': 'F1_uji'}))
celah['selisih'] = celah['F1_validasi'] - celah['F1_uji']
celah = celah.round(4)
display(celah) if 'display' in globals() else print(celah)
celah.to_csv(f'{OUT_DIR}/e1_selisih_validasi_uji.csv')

# Autokorelasi RESIDU (Roberts dkk., 2017): galat prediksi out-of-fold model terbaik pada data latih.
from sklearn.model_selection import cross_val_predict
m_cv = bangun(TERBAIK, SEED_UTAMA).set_params(**param_terbaik[KUNCI])
p_oof = cross_val_predict(m_cv, latih[SET_FITUR[SET_TERBAIK]].values, y_latih, groups=g_latih, cv=cv, n_jobs=-1)
dres = latih[['kode', 'petak', 'x', 'y']].copy(); dres['residu'] = (p_oof != y_latih).astype(float)
r_res = jangkauan(korelogram(dres, ['residu'])['residu'])
print(f'Jangkauan autokorelasi residu: {r_res:.0f} m; ukuran blok: {UKURAN_BLOK_M} m ->',
      'memadai' if r_res <= UKURAN_BLOK_M else 'blok LEBIH KECIL dari jangkauan residu; naikkan UKURAN_BLOK_M dan jalankan ulang')
pd.Series({'jangkauan_residu_m': r_res, 'ukuran_blok_m': UKURAN_BLOK_M}).to_csv(f'{OUT_DIR}/autokorelasi_residu.csv')

# Tabel ablasi: F1 makro rata-rata per set fitur x model
ablasi = hasil.groupby(['model', 'fitur'])['F1_makro'].mean().unstack()[list(SET_FITUR)].round(4)
ablasi.index = [NAMA_MODEL[m] for m in ablasi.index]
display(ablasi) if 'display' in globals() else print(ablasi)
ablasi.to_csv(f'{OUT_DIR}/e2_ablasi_f1_makro.csv')

In [ ]:
# Gambar ablasi: F1 makro per set fitur, satu garis per model (rata-rata seed; batang galat = SD)
fig, ax = plt.subplots(figsize=(7, 4.2))
stat = hasil.groupby(['model', 'fitur'])['F1_makro'].agg(['mean', 'std']).reset_index()
xs = np.arange(len(SET_FITUR))
penanda = {'LR': 'o', 'SVM': 's', 'RF': '^', 'HGB': 'D', 'MLP': 'v'}
for nama in GRID:
    s = stat[stat['model'] == nama].set_index('fitur').loc[list(SET_FITUR)]
    lebar = 2.4 if nama == TERBAIK else 1.2
    ax.errorbar(xs, s['mean'], yerr=s['std'].fillna(0), marker=penanda[nama], lw=lebar, capsize=3,
                label=NAMA_MODEL[nama])
ax.set_xticks(xs, ['FRGB\nRGB', 'F0\n10 pita', '+F1\nindeks', '+F2\ntekstur, tepi', '+F3\nmorfologi'])
ax.set_ylabel(f'F1 makro pada {UJI}')
ax.set_title('Pengaruh penambahan kelompok fitur PCD')
ax.grid(axis='y', alpha=0.3); ax.legend(fontsize=8, frameon=False, loc='lower right')
fig.tight_layout(); fig.savefig(f'{OUT_DIR}/gbr_ablasi_fitur.png', dpi=300); plt.show()

## 6. E3: Uji statistik

In [ ]:
def mcnemar(y, pa, pb):
    a_benar, b_benar = pa == y, pb == y
    n01 = int(np.sum(a_benar & ~b_benar)); n10 = int(np.sum(~a_benar & b_benar))
    if n01 + n10 == 0:
        return n01, n10, np.nan, 1.0
    if n01 + n10 < 25:                                   # uji eksak untuk sel kecil
        return n01, n10, np.nan, binomtest(min(n01, n10), n01 + n10, 0.5).pvalue
    z2 = (abs(n01 - n10) - 1) ** 2 / (n01 + n10)       # dengan koreksi kontinuitas
    return n01, n10, z2, chi2.sf(z2, 1)

def holm(p):
    p = np.asarray(p, float); urut = np.argsort(p); adj = np.empty_like(p); m = len(p); maks = 0
    for i, j in enumerate(urut):
        maks = max(maks, (m - i) * p[j]); adj[j] = min(1, maks)
    return adj

# Empat perbandingan yang direncanakan (satu keluarga uji, dikoreksi Holm):
#   1) terbaik vs kedua (CV)            : apakah pemenang benar-benar lebih baik
#   2) terbaik vs aturan PCD terbaik    : apakah ML diperlukan
#   3) set terbaik vs F0 (model sama)   : apakah fitur hasil PCD menambah akurasi
#   4) set terbaik vs FRGB (model sama) : apakah pita NIR/SWIR diperlukan
P_UJI = {KUNCI: prediksi[KUNCI], KEDUA: prediksi[KEDUA], ('PCD', PCD_TERBAIK): pred_pcd}
rencana = [('Terbaik vs kedua', KEDUA), ('Terbaik vs PCD terbaik', ('PCD', PCD_TERBAIK))]
for fs in ['F0', 'FRGB']:
    if SET_TERBAIK != fs:
        P_UJI[(TERBAIK, fs)] = prediksi[(TERBAIK, fs)]
        rencana.append((f'Set terbaik vs {fs} (model sama)', (TERBAIK, fs)))
    else:
        print(f'Perbandingan set terbaik vs {fs} dilewati: set terbaik adalah {fs}.')
nama_kunci = lambda k: f'PCD {k[1]}' if k[0] == 'PCD' else f'{k[0]} {k[1]} ({ID_ML[k]})'
uji_baris = []
for label, b in rencana:
    n01, n10, z2, p = mcnemar(y_uji, P_UJI[KUNCI], P_UJI[b])
    uji_baris.append({'perbandingan': label, 'model_A': nama_kunci(KUNCI), 'model_B': nama_kunci(b),
                      'n01_A_benar_B_salah': n01, 'n10_A_salah_B_benar': n10, 'z2': z2, 'p': p})

In [ ]:
# Bootstrap per blok: blok 200 m diambil ulang dengan pengembalian (menghormati autokorelasi spasial)
rng = np.random.RandomState(SEED_UTAMA)
blok_uji = uji['blok'].values
daftar_blok = np.unique(blok_uji)
indeks_blok = {b: np.where(blok_uji == b)[0] for b in daftar_blok}
f1m = lambda y, p: f1_score(y, p, labels=[0, 1, 2], average='macro', zero_division=0)
# Dibootstrap: kelima model pada set terbaik + semua prediksi yang dipakai uji McNemar + RGB terbaik.
for m in MODEL_URUT:
    P_UJI.setdefault((m, SET_TERBAIK), prediksi[(m, SET_TERBAIK)])
P_UJI.setdefault(RGB_TERBAIK, prediksi[RGB_TERBAIK])
boot = {k: [] for k in P_UJI}
for _ in range(N_BOOT):
    idx = np.concatenate([indeks_blok[b] for b in rng.choice(daftar_blok, len(daftar_blok), replace=True)])
    for k, p in P_UJI.items():
        boot[k].append(f1m(y_uji[idx], p[idx]))
ci = pd.DataFrame({nama_kunci(k): {'F1_makro': f1m(y_uji, P_UJI[k]),
                                   'CI95_bawah': np.percentile(v, 2.5), 'CI95_atas': np.percentile(v, 97.5)}
                   for k, v in boot.items()}).T.sort_values('F1_makro', ascending=False).round(4)
# Selisih F1 makro (A - B) dan CI-nya untuk setiap perbandingan McNemar.
for r, (label, b) in zip(uji_baris, rencana):
    d = np.array(boot[KUNCI]) - np.array(boot[b])
    r.update({'dF1_makro': f1m(y_uji, P_UJI[KUNCI]) - f1m(y_uji, P_UJI[b]),
              'dF1_CI95_bawah': np.percentile(d, 2.5), 'dF1_CI95_atas': np.percentile(d, 97.5)})
ujiM = pd.DataFrame(uji_baris)
ujiM['p_holm'] = holm(ujiM['p'])
ujiM['bermakna_5%'] = ujiM['p_holm'] < 0.05
display(ujiM.round(4)) if 'display' in globals() else print(ujiM.round(4))
ujiM.to_csv(f'{OUT_DIR}/e3_uji_mcnemar.csv', index=False)
display(ci) if 'display' in globals() else print(ci)

# Ringkasan T3 untuk spreadsheet tab 5_Uji_T3 (empat baris tetap).
def baris_t3(peran, k, skor_pilih):
    return {'peran': peran, 'id': f'PCD {k[1]}' if k[0] == 'PCD' else ID_ML[k], 'skor_pemilihan': skor_pilih,
            **metrik(y_uji, P_UJI[k]), 'CI95_bawah': np.percentile(boot[k], 2.5),
            'CI95_atas': np.percentile(boot[k], 97.5)}
t3 = pd.DataFrame([
    baris_t3('ML terbaik (peringkat CV 1)', KUNCI, tab_cv.loc[ID_ML[KUNCI], 'cv_F1_makro']),
    baris_t3('ML kedua (peringkat CV 2)', KEDUA, tab_cv.loc[ID_ML[KEDUA], 'cv_F1_makro']),
    baris_t3('PCD terbaik (F1 latih)', ('PCD', PCD_TERBAIK), e0.loc[PCD_TERBAIK, 'latih_F1_makro']),
    baris_t3('ML terbaik dengan FRGB', RGB_TERBAIK, tab_cv.loc[ID_ML[RGB_TERBAIK], 'cv_F1_makro']),
]).set_index('peran')
t3['selisih_pemilihan_T3'] = t3['skor_pemilihan'] - t3['F1_makro']
t3.round(4).to_csv(f'{OUT_DIR}/e3_uji_t3_untuk_spreadsheet.csv')
display(t3.round(4)) if 'display' in globals() else print(t3.round(4))
ci.to_csv(f'{OUT_DIR}/e3_bootstrap_ci.csv')

## 7. E4: Matriks konfusi, kepentingan fitur, uji silang tanggal, pembanding PCD

In [ ]:
p_best = prediksi[KUNCI]
cm = confusion_matrix(y_uji, p_best, labels=[0, 1, 2])
tabel_cm = pd.DataFrame(cm, index=[f'Ref: {KELAS[k]}' for k in range(3)],
                        columns=[f'Pred: {KELAS[k]}' for k in range(3)])
tabel_cm['Recall (%)'] = (np.diag(cm) / cm.sum(1).clip(min=1) * 100).round(1)
presisi = (np.diag(cm) / cm.sum(0).clip(min=1) * 100).round(1)
tabel_cm.loc['Presisi (%)'] = list(presisi) + [np.nan]
display(tabel_cm) if 'display' in globals() else print(tabel_cm)
tabel_cm.to_csv(f'{OUT_DIR}/e4_matriks_konfusi_{TERBAIK}.csv')

fig, ax = plt.subplots(figsize=(4.6, 4))
cm_pct = cm / cm.sum(1, keepdims=True).clip(min=1)
ax.imshow(cm_pct, cmap='Blues', vmin=0, vmax=1)
for i in range(3):
    for j in range(3):
        ax.text(j, i, f'{cm[i, j]}\n({cm_pct[i, j]*100:.0f}%)', ha='center', va='center', fontsize=9,
                color='white' if cm_pct[i, j] > 0.6 else 'black')
ax.set_xticks(range(3), [KELAS[k] for k in range(3)]); ax.set_yticks(range(3), [KELAS[k] for k in range(3)])
ax.set_xlabel('Prediksi'); ax.set_ylabel('Referensi'); ax.set_title(f'{NAMA_MODEL[TERBAIK]}, {UJI}')
fig.tight_layout(); fig.savefig(f'{OUT_DIR}/gbr_matriks_konfusi.png', dpi=300); plt.show()

In [ ]:
# Permutation importance pada data uji (penurunan F1 makro), per kelompok dan per fitur
m_best = model_akhir[KUNCI]
kolom = SET_FITUR[SET_TERBAIK]
X_uji_df = uji[kolom].reset_index(drop=True)
dasar = f1m(y_uji, m_best.predict(X_uji_df.values))
rng = np.random.RandomState(SEED_UTAMA)

def turun(cols):
    nilai = []
    for _ in range(N_PERMUTASI):
        X = X_uji_df.copy()
        perm = rng.permutation(len(X))
        X[cols] = X[cols].values[perm]                    # kolom satu kelompok diacak bersama
        nilai.append(dasar - f1m(y_uji, m_best.predict(X.values)))
    return np.mean(nilai), np.std(nilai)

kel_ada = {k: [c for c in v if c in kolom] for k, v in KELOMPOK.items()}
kel_ada = {k: v for k, v in kel_ada.items() if v}          # hanya kelompok yang ada di set terbaik
imp_kel = pd.DataFrame({k: dict(zip(['penurunan_F1', 'sd'], turun(v))) for k, v in kel_ada.items()}).T
imp_fit = pd.DataFrame({c: dict(zip(['penurunan_F1', 'sd'], turun([c]))) for c in kolom}).T
imp_fit = imp_fit.sort_values('penurunan_F1', ascending=False)
imp_kel.round(4).to_csv(f'{OUT_DIR}/e4_kepentingan_kelompok.csv'); imp_fit.round(4).to_csv(f'{OUT_DIR}/e4_kepentingan_fitur.csv')
display(imp_kel.round(4)) if 'display' in globals() else print(imp_kel.round(4))

warna_kel = {**{c: '#4c78a8' for c in F0}, **{c: '#f58518' for c in F1},
             **{c: '#54a24b' for c in F2}, **{c: '#b279a2' for c in F3}}
top = imp_fit.head(15)[::-1]
fig, ax = plt.subplots(figsize=(6, 5))
ax.barh(top.index, top['penurunan_F1'], xerr=top['sd'], color=[warna_kel[c] for c in top.index])
from matplotlib.patches import Patch
ax.legend(handles=[Patch(color=w, label=k) for k, w in [('F0 pita', '#4c78a8'), ('F1 indeks', '#f58518'),
                   ('F2 tekstur/tepi', '#54a24b'), ('F3 morfologi', '#b279a2')]],
          frameon=False, fontsize=8, loc='lower right')
ax.set_xlabel('Penurunan F1 makro bila fitur diacak'); ax.set_title(f'Kepentingan fitur {NAMA_MODEL[TERBAIK]}')
ax.grid(axis='x', alpha=0.3)
fig.tight_layout(); fig.savefig(f'{OUT_DIR}/gbr_kepentingan_fitur.png', dpi=300); plt.show()

In [ ]:
# Uji silang antartanggal (pemeriksaan internal) pada petak yang sama (A-C):
# latih T1 uji T2, dan sebaliknya.
silang = []
SAMA = [k for k in LATIH if k in ('T1', 'T2')]
if len(SAMA) >= 2:
    for a in SAMA:
        for b in SAMA:
            if a == b:
                continue
            la = latih[latih['kode'] == a]
            ub = df[(df['kode'] == b) & kelas_ok]
            if la['label'].nunique() < 3:
                continue
            m = bangun(TERBAIK, SEED_UTAMA).set_params(**param_terbaik[KUNCI])
            m.fit(la[SET_FITUR[SET_TERBAIK]].values, la['label'].values)
            silang.append({'latih': a, 'uji': b, **metrik(ub['label'].values, m.predict(ub[SET_FITUR[SET_TERBAIK]].values))})
silang.append({'latih': '+'.join(LATIH), 'uji': UJI, **metrik(y_uji, p_best)})
silang = pd.DataFrame(silang).round(4)
display(silang) if 'display' in globals() else print(silang)
silang.to_csv(f'{OUT_DIR}/e4_uji_silang_tanggal.csv', index=False)

In [ ]:
# Pembanding PCD: kelas air dari ambang indeks (air_pcd) vs kelas air dari model terbaik
y_air = (y_uji == 0).astype(int)
banding = pd.DataFrame([
    {'metode': 'Ambang indeks (PCD, tanpa pascaproses)', **dict(zip(['presisi', 'recall', 'F1'],
        precision_recall_fscore_support(y_air, uji['air_pcd'].values.astype(int), average='binary', zero_division=0)[:3]))},
    {'metode': f'{NAMA_MODEL[TERBAIK]} ({SET_TERBAIK})', **dict(zip(['presisi', 'recall', 'F1'],
        precision_recall_fscore_support(y_air, (p_best == 0).astype(int), average='binary', zero_division=0)[:3]))},
]).round(4)
cakupan = (uji['waduk_pcd'] == 1).mean()
print(f'Piksel uji di dalam masker waduk PCD: {cakupan*100:.1f}% (sisanya tidak akan terpetakan pada peta seluruh waduk)')
display(banding) if 'display' in globals() else print(banding)
banding.to_csv(f'{OUT_DIR}/e4_pembanding_pcd_kelas_air.csv', index=False)

# Bias proporsi kelas: bobot kelas seimbang bisa menggeser proporsi kelas hasil prediksi
# (Millard & Richardson, 2015). Bandingkan proporsi referensi dan prediksi di T3.
prop = pd.DataFrame({
    'referensi_%': pd.Series(y_uji).map(KELAS).value_counts(normalize=True) * 100,
    'prediksi_%': pd.Series(p_best).map(KELAS).value_counts(normalize=True) * 100,
}).fillna(0)
prop['bias_poin'] = prop['prediksi_%'] - prop['referensi_%']
display(prop.round(2)) if 'display' in globals() else print(prop.round(2))
prop.round(3).to_csv(f'{OUT_DIR}/e4_bias_proporsi_kelas.csv')
print('Luas dari peta adalah luas terpetakan, bukan estimasi luas tak bias (Olofsson dkk., 2014).')

## 8. E5: Pengaruh petak latih tambahan
Model terbaik (hiperparameter sama) dilatih dua kali: (a) hanya petak A–C pada T1, T2; (b) ditambah petak D, E, F pada D1, D2, D3. Keduanya diuji di T3. Bila (b) lebih baik dan bermakna, keragaman lokasi dan waktu terbukti membantu generalisasi.

In [ ]:
tambahan_ada = latih['petak'].isin(PETAK_TAMBAHAN).any()
if not tambahan_ada:
    print('Data latih tidak memuat petak tambahan; E5 dilewati.')
else:
    kol = SET_FITUR[SET_TERBAIK]
    hasil_e5, pred_e5 = [], {}
    for nama_set, bagian in [('A-C saja', latih[~latih['petak'].isin(PETAK_TAMBAHAN)]), ('A-C + D-F', latih)]:
        for seed in (SEEDS if TERBAIK in ACAK else [SEED_UTAMA]):
            m = bangun(TERBAIK, seed).set_params(**param_terbaik[KUNCI])
            m.fit(bagian[kol].values, bagian['label'].values)
            p = m.predict(uji[kol].values)
            hasil_e5.append({'data_latih': nama_set, 'seed': seed, 'n_latih': len(bagian), **metrik(y_uji, p)})
            if seed == SEED_UTAMA:
                pred_e5[nama_set] = p
        # Skor validasi silang skenario ini (GroupKFold blok, hiperparameter sama).
        from sklearn.model_selection import cross_val_score
        cv_e5 = GroupKFold(n_splits=min(N_FOLD, bagian['blok'].nunique()))
        skor = cross_val_score(bangun(TERBAIK, SEED_UTAMA).set_params(**param_terbaik[KUNCI]),
                               bagian[kol].values, bagian['label'].values, groups=bagian['blok'].values,
                               cv=cv_e5, scoring='f1_macro', n_jobs=-1)
        for r in hasil_e5:
            if r['data_latih'] == nama_set:
                r['cv_F1_makro'] = skor.mean()
    e5 = pd.DataFrame(hasil_e5).groupby('data_latih')[['n_latih', 'cv_F1_makro'] + kolom_m].agg(['mean', 'std']).round(4)
    n01, n10, z2, pval = mcnemar(y_uji, pred_e5['A-C + D-F'], pred_e5['A-C saja'])
    print(f'McNemar (A-C + D-F) vs (A-C saja): benar_A_salah_B={n01}, salah_A_benar_B={n10}, p={pval:.4f}')
    display(e5) if 'display' in globals() else print(e5)
    e5.to_csv(f'{OUT_DIR}/e5_petak_tambahan.csv')

## 9. Simpan model dan ringkasan

In [ ]:
import joblib
joblib.dump({'model': m_best, 'fitur': SET_FITUR[SET_TERBAIK], 'kelas': KELAS, 'log1p': ['luas_komponen', 'jarak_air']},
            f'{OUT_DIR}/model_terbaik_{TERBAIK}.joblib')
ringkas = {
    'lingkungan': LINGKUNGAN, 'versi_pustaka': VERSI, 'versi_dataset': VERSI_DATASET,
    'latih': LATIH, 'uji': UJI, 'uji_coba': bool(UJI_COBA), 'min_sepakat_latih': MIN_SEPAKAT_LATIH, 'buang_eg_petak_c': BUANG_EG_PETAK_C,
    'n_latih': int(len(latih)), 'n_uji': int(len(uji)), 'model_terbaik': TERBAIK,
    'hiperparameter_terbaik': {f'{k[0]}|{k[1]}': {kk: str(vv) for kk, vv in v.items()} for k, v in param_terbaik.items()},
    'set_fitur_terbaik': SET_TERBAIK, 'id_terbaik': ID_ML[KUNCI],
    'cv_f1_makro_terbaik': float(tab_cv.loc[ID_ML[KUNCI], 'cv_F1_makro']),
    'f1_makro_terbaik_T3': float(f1m(y_uji, P_UJI[KUNCI])),
    'pcd_terbaik': PCD_TERBAIK, 'ambang_pcd': {k: float(v) for k, v in AMBANG_PCD.items()},
}
json.dump(ringkas, open(f'{OUT_DIR}/ringkasan.json', 'w'), indent=2, ensure_ascii=False)
print('Tersimpan di', OUT_DIR); print(sorted(os.listdir(OUT_DIR)))

## 10. (Opsional) Peta dan luas seluruh waduk
Membutuhkan `waduk_<UJI>_<versi>.csv` dari Langkah 10 mode `ekspor_waduk`. Luas dihitung dari jumlah piksel × 0,04 ha. Piksel di luar masker waduk PCD dan area PLTS sudah dikeluarkan di GEE.

In [ ]:
p_waduk = cari(f'waduk_{UJI}_{VERSI_DATASET}.csv')
if p_waduk is None:
    print(f'waduk_{UJI}_{VERSI_DATASET}.csv tidak ditemukan; bagian ini dilewati.')
else:
    w = pd.read_csv(p_waduk)
    for c in ['luas_komponen', 'jarak_air']:
        w[c] = np.log1p(w[c].clip(lower=0))
    w['prediksi'] = m_best.predict(w[SET_FITUR[SET_TERBAIK]].values)
    luas = w['prediksi'].map(KELAS).value_counts().mul(0.04).round(1).rename('luas_ha')
    display(luas.to_frame()) if 'display' in globals() else print(luas)
    luas.to_csv(f'{OUT_DIR}/luas_kelas_seluruh_waduk_{UJI}.csv')

    # Raster dari koordinat piksel (x, y = pusat piksel UTM 48S, grid 20 m)
    xs, ys = np.sort(w['x'].unique()), np.sort(w['y'].unique())[::-1]
    kol = np.round((w['x'] - xs.min()) / 20).astype(int); bar = np.round((ys.max() - w['y']) / 20).astype(int)
    grid = np.full((bar.max() + 1, kol.max() + 1), -1, dtype=np.int8); grid[bar, kol] = w['prediksi']
    from matplotlib.colors import ListedColormap
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(np.ma.masked_less(grid, 0), cmap=ListedColormap(['#2166ac', '#e7298a', '#fdb863']),
              vmin=0, vmax=2, interpolation='nearest')
    for k, c in [(0, '#2166ac'), (1, '#e7298a'), (2, '#fdb863')]:
        ax.plot([], [], 's', color=c, label=f'{KELAS[k]} ({luas.get(KELAS[k], 0)} ha)')
    ax.legend(loc='lower left', frameon=True, fontsize=8); ax.set_axis_off()
    ax.set_title(f'Klasifikasi permukaan Waduk Saguling, {UJI} ({NAMA_MODEL[TERBAIK]})')
    fig.tight_layout(); fig.savefig(f'{OUT_DIR}/gbr_peta_seluruh_waduk_{UJI}.png', dpi=300); plt.show()
    try:
        import rasterio
        from rasterio.transform import from_origin
        with rasterio.open(f'{OUT_DIR}/peta_{UJI}_{TERBAIK}.tif', 'w', driver='GTiff', height=grid.shape[0],
                           width=grid.shape[1], count=1, dtype='int8', crs='EPSG:32748', nodata=-1,
                           transform=from_origin(xs.min() - 10, ys.max() + 10, 20, 20)) as dst:
            dst.write(grid, 1)
        print('GeoTIFF tersimpan (bisa dibuka di QGIS).')
    except ImportError:
        print('rasterio tidak terpasang; GeoTIFF dilewati (PNG tetap tersimpan). Colab: !pip install rasterio')